# Challenge 5 — Group 3: Public Health & Epidemiology
## Notebook 02 — Preprocessing & Feature Engineering

**Goal:** Transform the raw BRFSS sample into a clean, scaled feature matrix
ready for K-Means, DBSCAN, and Hierarchical Clustering.

**Pipeline overview:**
1. Imports
2. Load Clean Dataset
3. Feature Selection
4. Missing Value Imputation
5. Feature Encoding
6. Feature Scaling
7. PCA Dimensionality Reduction
8. Save Processed Data



## 1. Imports, Configuration 

In [24]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

# Dimensionality Reduction
from sklearn.decomposition import PCA

# Pipeline utilities
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Save models
import joblib

# Random seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded successfully.")

Libraries loaded successfully.



## 2. Load Sampled Data

In [25]:
# =========================
# Load datasets
# =========================

DATA_PATH_30K = "../data/brfss_sample_30k.csv"
DATA_PATH_5K = "../data/brfss_sample_5k.csv"

df_30k = pd.read_csv(DATA_PATH_30K)
df_5k = pd.read_csv(DATA_PATH_5K)

print("30k dataset shape:", df_30k.shape)
print("5k dataset shape:", df_5k.shape)

30k dataset shape: (29999, 19)
5k dataset shape: (5000, 19)


## Preview dataset

In [26]:


df_30k.head()

,smoking_status,alcohol_any,binge_drinking,physical_activity,bmi,diabetes,coronary_heart_disease,asthma,copd,depression,last_checkup,flu_vaccine,pneumo_vaccine,age_group,sex,education,income_group,race,state
0,1.0,2.0,NaN,1.0,27.12,1.0,1.0,2.0,1.0,2.0,1.0,1.0,1.0,11.0,2.0,4.0,4.0,1.0,50.0
1,4.0,2.0,NaN,1.0,21.93,3.0,2.0,2.0,2.0,2.0,1.0,1.0,1.0,10.0,1.0,3.0,4.0,1.0,53.0
2,4.0,2.0,NaN,1.0,19.94,3.0,2.0,2.0,2.0,2.0,1.0,2.0,NaN,1.0,1.0,1.0,7.0,2.0,20.0
3,4.0,1.0,5.0,1.0,30.41,3.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,8.0,1.0,4.0,6.0,1.0,53.0
4,2.0,2.0,NaN,1.0,23.75,3.0,2.0,1.0,2.0,2.0,1.0,2.0,1.0,4.0,1.0,3.0,9.0,2.0,18.0


---
## 3. Define Feature Groups

We separate clustering features from demographic features.
Demographics are carried along but **never fed to the clustering algorithms**.

In [27]:
# =========================
# Dataset information
# =========================

print("30k Dataset Info")
print("=" * 50)

display(df_30k.info())

print("\nMissing values:")
display(df_30k.isnull().sum().sort_values(ascending=False).head(20))

30k Dataset Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29999 entries, 0 to 29998
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   smoking_status          29999 non-null  float64
 1   alcohol_any             29048 non-null  float64
 2   binge_drinking          14529 non-null  float64
 3   physical_activity       29911 non-null  float64
 4   bmi                     28177 non-null  float64
 5   diabetes                29938 non-null  float64
 6   coronary_heart_disease  29705 non-null  float64
 7   asthma                  29891 non-null  float64
 8   copd                    29861 non-null  float64
 9   depression              29825 non-null  float64
 10  last_checkup            29661 non-null  float64
 11  flu_vaccine             28988 non-null  float64
 12  pneumo_vaccine          26941 non-null  float64
 13  age_group               29999 non-null  float64
 14  sex                  

None


Missing values:


binge_drinking            15470
pneumo_vaccine             3058
bmi                        1822
flu_vaccine                1011
alcohol_any                 951
last_checkup                338
coronary_heart_disease      294
depression                  174
copd                        138
asthma                      108
physical_activity            88
diabetes                     61
smoking_status                0
age_group                     0
sex                           0
education                     0
income_group                  0
race                          0
state                         0
dtype: int64

In [28]:
# =========================
# Column names
# =========================

print(df_30k.columns.tolist())

['smoking_status', 'alcohol_any', 'binge_drinking', 'physical_activity', 'bmi', 'diabetes', 'coronary_heart_disease', 'asthma', 'copd', 'depression', 'last_checkup', 'flu_vaccine', 'pneumo_vaccine', 'age_group', 'sex', 'education', 'income_group', 'race', 'state']


In [32]:
# =========================
# Feature categorization
# =========================

# Numerical continuous variables
numerical_features = [
    "bmi"
]

# Binary variables
binary_features = [
    "smoking_status",
    "alcohol_any",
    "binge_drinking",
    "physical_activity",
    "diabetes",
    "coronary_heart_disease",
    "asthma",
    "copd",
    "depression",
    "flu_vaccine",
    "pneumo_vaccine",
    "sex"
]

# Categorical / ordinal variables
categorical_features = [
    "last_checkup",
    "age_group",
    "education",
    "income_group",
    "race",    
]

# Combine all selected features
selected_features = (
    numerical_features
    + binary_features
    + categorical_features
)

print("Total selected features:", len(selected_features))
print(selected_features)

Total selected features: 18
['bmi', 'smoking_status', 'alcohol_any', 'binge_drinking', 'physical_activity', 'diabetes', 'coronary_heart_disease', 'asthma', 'copd', 'depression', 'flu_vaccine', 'pneumo_vaccine', 'sex', 'last_checkup', 'age_group', 'education', 'income_group', 'race']


In [30]:
# =========================
# Validate feature existence
# =========================

missing_cols = [col for col in selected_features if col not in df_30k.columns]

if len(missing_cols) == 0:
    print("All selected features exist.")
else:
    print("Missing columns:")
    print(missing_cols)


All selected features exist.


In [33]:
# =========================
# Subset dataset
# =========================

df_selected = df_30k[selected_features].copy()

print("Selected dataset shape:", df_selected.shape)

df_selected.head()

Selected dataset shape: (29999, 18)


,bmi,smoking_status,alcohol_any,binge_drinking,physical_activity,diabetes,coronary_heart_disease,asthma,copd,depression,flu_vaccine,pneumo_vaccine,sex,last_checkup,age_group,education,income_group,race
0,27.12,1.0,2.0,NaN,1.0,1.0,1.0,2.0,1.0,2.0,1.0,1.0,2.0,1.0,11.0,4.0,4.0,1.0
1,21.93,4.0,2.0,NaN,1.0,3.0,2.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,10.0,3.0,4.0,1.0
2,19.94,4.0,2.0,NaN,1.0,3.0,2.0,2.0,2.0,2.0,2.0,NaN,1.0,1.0,1.0,1.0,7.0,2.0
3,30.41,4.0,1.0,5.0,1.0,3.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,8.0,4.0,6.0,1.0
4,23.75,2.0,2.0,NaN,1.0,3.0,2.0,1.0,2.0,2.0,2.0,1.0,1.0,1.0,4.0,3.0,9.0,2.0


---
## 4. PRE-Processing Data

We will use simple imputation

In [34]:
# =========================
# Final feature groups
# =========================

numerical_features = [
    "bmi"
]

binary_features = [
    'alcohol_any',
    'binge_drinking',
    'physical_activity',
    'diabetes',
    'coronary_heart_disease',
    'asthma',
    'copd',
    'depression',
    'flu_vaccine',
    'pneumo_vaccine',
    'sex'
]

categorical_features = [
    'smoking_status',
    'last_checkup',
    'age_group',
    'education',
    'income_group',
    'race'
]

selected_features = (
    numerical_features
    + binary_features
    + categorical_features
)

In [35]:
# =========================
# Subset final dataset
# =========================

df_selected = df_30k[selected_features].copy()

print(df_selected.shape)

df_selected.head()

(29999, 18)


,bmi,alcohol_any,binge_drinking,physical_activity,diabetes,coronary_heart_disease,asthma,copd,depression,flu_vaccine,pneumo_vaccine,sex,smoking_status,last_checkup,age_group,education,income_group,race
0,27.12,2.0,NaN,1.0,1.0,1.0,2.0,1.0,2.0,1.0,1.0,2.0,1.0,1.0,11.0,4.0,4.0,1.0
1,21.93,2.0,NaN,1.0,3.0,2.0,2.0,2.0,2.0,1.0,1.0,1.0,4.0,1.0,10.0,3.0,4.0,1.0
2,19.94,2.0,NaN,1.0,3.0,2.0,2.0,2.0,2.0,2.0,NaN,1.0,4.0,1.0,1.0,1.0,7.0,2.0
3,30.41,1.0,5.0,1.0,3.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,4.0,1.0,8.0,4.0,6.0,1.0
4,23.75,2.0,NaN,1.0,3.0,2.0,1.0,2.0,2.0,2.0,1.0,1.0,2.0,1.0,4.0,3.0,9.0,2.0


In [36]:
# =========================
# Missing values check
# =========================

missing_summary = (
    df_selected
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

display(missing_summary)

binge_drinking            15470
pneumo_vaccine             3058
bmi                        1822
flu_vaccine                1011
alcohol_any                 951
last_checkup                338
coronary_heart_disease      294
depression                  174
copd                        138
asthma                      108
physical_activity            88
diabetes                     61
sex                           0
smoking_status                0
age_group                     0
education                     0
income_group                  0
race                          0
dtype: int64

In [37]:
# =========================
# Preprocessing Pipelines
# =========================

# Numerical pipeline
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Binary pipeline
binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler", StandardScaler())
])

# Categorical pipeline
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [38]:
# =========================
# Column transformer
# =========================

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("bin", binary_pipeline, binary_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [39]:
# =========================
# Fit and transform
# =========================

X_processed = preprocessor.fit_transform(df_selected)

print("Processed matrix shape:", X_processed.shape)

Processed matrix shape: (29999, 51)


In [40]:
# Convert sparse matrix to dense if needed

if hasattr(X_processed, "toarray"):
    X_processed = X_processed.toarray()

print(type(X_processed))
print(X_processed.shape)

<class 'numpy.ndarray'>
(29999, 51)


---
## 5. Step 2 — Drop High-Missingness Features

Features with >40% missing values provide more noise than signal.
We drop them from both samples.

In [9]:
MISSING_THRESHOLD = 40.0  # percent

high_missing = missing_pct[missing_pct > MISSING_THRESHOLD].index.tolist()

if high_missing:
    print(f'Dropping {len(high_missing)} features with >{MISSING_THRESHOLD}% missing:')
    for col in high_missing:
        print(f'  • {col}: {missing_pct[col]:.1f}%')
    CLUSTER_COLS = [c for c in CLUSTER_COLS if c not in high_missing]
else:
    print('No features exceed the missingness threshold ✅')

print(f'\nFinal clustering feature count: {len(CLUSTER_COLS)}')
print(f'Features retained: {CLUSTER_COLS}')

Dropping 2 features with >40.0% missing:
  • bmi: 100.0%
  • binge_drinking: 51.6%

Final clustering feature count: 11
Features retained: ['smoking_status', 'alcohol_any', 'physical_activity', 'diabetes', 'coronary_heart_disease', 'asthma', 'copd', 'depression', 'last_checkup', 'flu_vaccine', 'pneumo_vaccine']


---
## 6. Step 3 — Binary Encoding

BRFSS yes/no variables use the coding: **1 = Yes, 2 = No**.  
We recode these to standard binary: **1 = Yes, 0 = No**.

Multi-category variables (smoking_status, last_checkup, diabetes) are
kept as ordered integers — they already carry a meaningful scale.

In [10]:
# Variables coded as 1=yes, 2=no → recode to 1=yes, 0=no
YES_NO_COLS = [
    'alcohol_any', 'physical_activity', 'hypertension',
    'high_cholesterol', 'coronary_heart_disease', 'asthma',
    'copd', 'depression', 'flu_vaccine', 'pneumo_vaccine'
]

def encode_binary(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            # 1 → 1 (yes), 2 → 0 (no), NaN stays NaN
            df[col] = df[col].map({1.0: 1, 2.0: 0})
    return df

df30 = encode_binary(df30, YES_NO_COLS)
df5  = encode_binary(df5,  YES_NO_COLS)

# Recode smoking_status: 1=daily, 2=someday, 3=former, 4=never
# Keep as-is (ordinal from most to least active smoker)

# Recode diabetes: 1=yes, 2=yes(borderline), 3=no, 4=gestational only
# Simplify to: 1=yes/pre-diabetes, 0=no
for df in [df30, df5]:
    if 'diabetes' in df.columns:
        df['diabetes'] = df['diabetes'].map({1.0: 1, 2.0: 1, 3.0: 0, 4.0: 0})

print('Binary encoding complete ✅')
print('\nSample value counts — physical_activity:')
print(df30['physical_activity'].value_counts())

Binary encoding complete ✅

Sample value counts — physical_activity:
physical_activity
1.0    23014
0.0     6897
Name: count, dtype: int64


---
## 7. Step 4 — Cap Continuous Outliers

Extreme outliers distort distance-based clustering (K-Means, DBSCAN).
We cap continuous variables at the **1st and 99th percentile** (Winsorization).

In [11]:
CONTINUOUS_COLS = ['bmi', 'sleep_hours', 'binge_drinking', 'fruit_per_day', 'veg_per_day']
CONTINUOUS_COLS = [c for c in CONTINUOUS_COLS if c in CLUSTER_COLS]

def winsorize(df, cols, lower=0.01, upper=0.99):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            lo = df[col].quantile(lower)
            hi = df[col].quantile(upper)
            before = df[col].describe()[['min','max']].to_dict()
            df[col] = df[col].clip(lower=lo, upper=hi)
            after  = df[col].describe()[['min','max']].to_dict()
            print(f'  {col:20s}  [{before["min"]:.1f}, {before["max"]:.1f}]'
                  f' → [{after["min"]:.1f}, {after["max"]:.1f}]')
    return df

print('Winsorizing (1st–99th percentile):')
df30 = winsorize(df30, CONTINUOUS_COLS)
df5  = winsorize(df5,  CONTINUOUS_COLS)
print('\nOutlier capping complete ✅')

Winsorizing (1st–99th percentile):

Outlier capping complete ✅


---
## 8. Step 5 — Impute Remaining NaNs

After recoding and capping, some NaNs remain. We impute them:
- **Continuous features** → median imputation (robust to skew)
- **Categorical/binary features** → mode imputation

We fit the imputer **only on the 30k sample** and apply the same
transformation to the 5k sample to avoid data leakage.

In [12]:
BINARY_COLS = [c for c in CLUSTER_COLS if c not in CONTINUOUS_COLS]

imputer_median = SimpleImputer(strategy='median')
imputer_mode   = SimpleImputer(strategy='most_frequent')

# Only keep columns that actually exist in the dataframe
cont_present = [c for c in CONTINUOUS_COLS if c in CLUSTER_COLS and c in df30.columns]
bin_present  = [c for c in BINARY_COLS     if c in CLUSTER_COLS and c in df30.columns]

# ── Diagnostic: see what's in each list ──────────────────────────────────────
print(f'Continuous cols to impute ({len(cont_present)}): {cont_present}')
print(f'Binary cols to impute     ({len(bin_present)}): {bin_present}')

# ── Safe imputation (skips empty lists) ──────────────────────────────────────
def safe_impute(df, cont_cols, bin_cols, fit=True,
                imp_med=None, imp_mod=None):
    df = df.copy()

    if cont_cols:
        if fit:
            df[cont_cols] = imp_med.fit_transform(df[cont_cols])
        else:
            df[cont_cols] = imp_med.transform(df[cont_cols])
    else:
        print('  ⚠️  No continuous columns to impute — skipping median imputer.')

    if bin_cols:
        if fit:
            df[bin_cols] = imp_mod.fit_transform(df[bin_cols])
        else:
            df[bin_cols] = imp_mod.transform(df[bin_cols])
    else:
        print('  ⚠️  No binary columns to impute — skipping mode imputer.')

    return df

df30_imp = safe_impute(df30, cont_present, bin_present,
                        fit=True, imp_med=imputer_median, imp_mod=imputer_mode)
df5_imp  = safe_impute(df5,  cont_present, bin_present,
                        fit=False, imp_med=imputer_median, imp_mod=imputer_mode)

print(f'\nMissing values remaining — 30k: {df30_imp[CLUSTER_COLS].isnull().sum().sum()}')
print(f'Missing values remaining — 5k : {df5_imp[CLUSTER_COLS].isnull().sum().sum()}')
print('Imputation complete ✅')

Continuous cols to impute (0): []
Binary cols to impute     (11): ['smoking_status', 'alcohol_any', 'physical_activity', 'diabetes', 'coronary_heart_disease', 'asthma', 'copd', 'depression', 'last_checkup', 'flu_vaccine', 'pneumo_vaccine']
  ⚠️  No continuous columns to impute — skipping median imputer.
  ⚠️  No continuous columns to impute — skipping median imputer.

Missing values remaining — 30k: 0
Missing values remaining — 5k : 0
Imputation complete ✅


---
## 9. Step 6 — StandardScaler Normalization

**This is mandatory for K-Means and DBSCAN** — both are distance-based.
Without scaling, high-variance features (e.g., BMI range 15–60) would
completely dominate over binary features (range 0–1).

StandardScaler transforms each feature to **mean=0, std=1**.

We fit on the 30k sample and apply to both samples.

In [13]:
scaler = StandardScaler()

# Extract only the clustering columns
X30_raw = df30_imp[CLUSTER_COLS].values
X5_raw  = df5_imp[CLUSTER_COLS].values

# Fit on 30k, transform both
X30_scaled = scaler.fit_transform(X30_raw)
X5_scaled  = scaler.transform(X5_raw)

print(f'X30_scaled shape : {X30_scaled.shape}  (K-Means & DBSCAN)')
print(f'X5_scaled  shape : {X5_scaled.shape}   (Hierarchical Clustering)')
print(f'\nMean per feature (should be ~0): {X30_scaled.mean(axis=0).round(4)}')
print(f'Std  per feature (should be ~1): {X30_scaled.std(axis=0).round(4)}')

X30_scaled shape : (29999, 11)  (K-Means & DBSCAN)
X5_scaled  shape : (5000, 11)   (Hierarchical Clustering)

Mean per feature (should be ~0): [ 0.  0.  0.  0. -0.  0.  0.  0.  0. -0. -0.]
Std  per feature (should be ~1): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [14]:
print('=== DIAGNOSTIC ===')
print(f'CONTINUOUS_COLS : {CONTINUOUS_COLS}')
print(f'CLUSTER_COLS    : {CLUSTER_COLS}')
print()
print('Checking which continuous cols are in CLUSTER_COLS:')
for c in CONTINUOUS_COLS:
    in_cluster = c in CLUSTER_COLS
    in_df30    = c in df30.columns
    print(f'  {c:20s} → in CLUSTER_COLS: {in_cluster}, in df30: {in_df30}')

=== DIAGNOSTIC ===
CONTINUOUS_COLS : []
CLUSTER_COLS    : ['smoking_status', 'alcohol_any', 'physical_activity', 'diabetes', 'coronary_heart_disease', 'asthma', 'copd', 'depression', 'last_checkup', 'flu_vaccine', 'pneumo_vaccine']

Checking which continuous cols are in CLUSTER_COLS:


In [15]:
# Visualise before vs after scaling for continuous features
fig, axes = plt.subplots(2, len(cont_present), figsize=(4*len(cont_present), 7))

for i, col in enumerate(cont_present):
    col_idx = CLUSTER_COLS.index(col)

    # Before scaling
    axes[0, i].hist(X30_raw[:, col_idx], bins=40, color='steelblue',
                    edgecolor='white', alpha=0.8)
    axes[0, i].set_title(f'{col}\n(raw)', fontsize=10)
    axes[0, i].set_ylabel('Count' if i == 0 else '')

    # After scaling
    axes[1, i].hist(X30_scaled[:, col_idx], bins=40, color='darkorange',
                    edgecolor='white', alpha=0.8)
    axes[1, i].set_title(f'{col}\n(scaled)', fontsize=10)
    axes[1, i].set_ylabel('Count' if i == 0 else '')

plt.suptitle('Before vs After StandardScaler — Continuous Features', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIG_PATH}scaling_comparison.png', bbox_inches='tight')
plt.show()
print('Figure saved: figures/scaling_comparison.png')

ValueError: Number of columns must be a positive integer, not 0

<Figure size 0x840 with 0 Axes>

---
## 10. Step 7 — PCA Dimensionality Reduction

We apply PCA for two purposes:
1. **Noise reduction** — retain ≥90% of variance, discard noise components
2. **Visualization** — project to 2D for all cluster scatter plots

The challenge requires comparing clustering on:
- Full scaled matrix (`X_scaled`)
- PCA-reduced matrix (`X_pca`)

> We fit PCA on the 30k sample and apply the same transform to the 5k sample.

In [ ]:
# ── Fit PCA on 30k sample ─────────────────────────────────────────────────────
pca_full = PCA(random_state=RANDOM_SEED)
pca_full.fit(X30_scaled)

# Cumulative explained variance
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

# Find number of components for ≥90% variance
n_components_90 = np.argmax(cumvar >= 0.90) + 1
print(f'Components needed for ≥90% variance: {n_components_90}')
print(f'Total features                      : {X30_scaled.shape[1]}')
print(f'Dimensionality reduction            : {X30_scaled.shape[1]} → {n_components_90}')

# ── Plot explained variance ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(range(1, len(cumvar)+1),
       pca_full.explained_variance_ratio_ * 100,
       color='steelblue', alpha=0.7, label='Individual component')
ax2 = ax.twinx()
ax2.plot(range(1, len(cumvar)+1), cumvar * 100,
         color='red', marker='o', markersize=4, label='Cumulative variance')
ax2.axhline(90, color='red', linestyle='--', alpha=0.5, label='90% threshold')
ax2.axvline(n_components_90, color='green', linestyle='--', alpha=0.7,
            label=f'{n_components_90} components')

ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance (%)', color='steelblue')
ax2.set_ylabel('Cumulative Variance (%)', color='red')
ax2.set_ylim(0, 105)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='center right')
ax.set_title('PCA — Explained Variance per Component', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIG_PATH}pca_explained_variance.png', bbox_inches='tight')
plt.show()
print('Figure saved: figures/pca_explained_variance.png')

In [ ]:
# ── Apply PCA with chosen number of components ────────────────────────────────
pca = PCA(n_components=n_components_90, random_state=RANDOM_SEED)
X30_pca = pca.fit_transform(X30_scaled)
X5_pca  = pca.transform(X5_scaled)

# 2D projection for visualization (always 2 components)
pca_2d = PCA(n_components=2, random_state=RANDOM_SEED)
X30_2d = pca_2d.fit_transform(X30_scaled)
X5_2d  = pca_2d.transform(X5_scaled)

print(f'X30_pca shape (≥90% var) : {X30_pca.shape}')
print(f'X5_pca  shape (≥90% var) : {X5_pca.shape}')
print(f'X30_2d  shape (2D visual): {X30_2d.shape}')
print(f'\nVariance retained (PCA reduced): {pca.explained_variance_ratio_.sum()*100:.1f}%')
print(f'Variance retained (2D visual)  : {pca_2d.explained_variance_ratio_.sum()*100:.1f}%')

---
## 11. PCA Component Interpretation

Understanding what each principal component represents
is essential for interpreting cluster visualizations.

In [ ]:
# ── Feature loadings for PC1 and PC2 ─────────────────────────────────────────
loadings = pd.DataFrame(
    pca_2d.components_.T,
    index=CLUSTER_COLS,
    columns=['PC1', 'PC2']
).round(3)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

for ax, pc in zip(axes, ['PC1', 'PC2']):
    sorted_load = loadings[pc].sort_values()
    colors = ['#e74c3c' if v > 0 else '#3498db' for v in sorted_load]
    sorted_load.plot(kind='barh', ax=ax, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{pc} Feature Loadings\n'
                 f'(explains {pca_2d.explained_variance_ratio_[int(pc[-1])-1]*100:.1f}% variance)',
                 fontsize=11)
    ax.set_xlabel('Loading')

plt.suptitle('PCA Feature Loadings — PC1 and PC2', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIG_PATH}pca_loadings.png', bbox_inches='tight')
plt.show()

print('\nTop contributors to PC1 (positive):')
print(loadings['PC1'].nlargest(5).to_string())
print('\nTop contributors to PC1 (negative):')
print(loadings['PC1'].nsmallest(5).to_string())

---
## 12. Save All Preprocessed Matrices

We save every matrix and object needed by the clustering notebooks.

In [ ]:
# ── Save numpy arrays ─────────────────────────────────────────────────────────
np.save(f'{DATA_OUT}X30_scaled.npy',  X30_scaled)   # full scaled — K-Means & DBSCAN
np.save(f'{DATA_OUT}X30_pca.npy',     X30_pca)      # PCA reduced — K-Means & DBSCAN
np.save(f'{DATA_OUT}X30_2d.npy',      X30_2d)       # 2D — all visualizations
np.save(f'{DATA_OUT}X5_scaled.npy',   X5_scaled)    # full scaled — Hierarchical
np.save(f'{DATA_OUT}X5_pca.npy',      X5_pca)       # PCA reduced — Hierarchical
np.save(f'{DATA_OUT}X5_2d.npy',       X5_2d)        # 2D — Hierarchical visualization

# ── Save scalers and PCA objects (needed to transform new data) ───────────────
joblib.dump(scaler,    f'{DATA_OUT}scaler.pkl')
joblib.dump(pca,       f'{DATA_OUT}pca.pkl')
joblib.dump(pca_2d,    f'{DATA_OUT}pca_2d.pkl')
joblib.dump(imputer_median, f'{DATA_OUT}imputer_median.pkl')
joblib.dump(imputer_mode,   f'{DATA_OUT}imputer_mode.pkl')

# ── Save demographic columns for interpretation ───────────────────────────────
df30_imp[DEMO_COLS].to_csv(f'{DATA_OUT}demo_30k.csv', index=False)
df5_imp[DEMO_COLS].to_csv(f'{DATA_OUT}demo_5k.csv',   index=False)

# ── Save feature names ────────────────────────────────────────────────────────
import json
with open(f'{DATA_OUT}cluster_cols.json', 'w') as f:
    json.dump(CLUSTER_COLS, f)

print('All matrices saved ✅')
print(f'\nSummary of saved files:')
print(f'  X30_scaled  : {X30_scaled.shape}  → K-Means & DBSCAN (full features)')
print(f'  X30_pca     : {X30_pca.shape}     → K-Means & DBSCAN (PCA reduced)')
print(f'  X30_2d      : {X30_2d.shape}  → 2D visualization')
print(f'  X5_scaled   : {X5_scaled.shape}   → Hierarchical (full features)')
print(f'  X5_pca      : {X5_pca.shape}      → Hierarchical (PCA reduced)')

---
## 13. Preprocessing Summary

This cell prints the full pipeline summary for your IEEE paper.

In [ ]:
print('=' * 65)
print('PREPROCESSING SUMMARY — Challenge 5, Group 3')
print('=' * 65)
print(f"""
Input  : CDC BRFSS 2023 — stratified samples
         30k rows (K-Means & DBSCAN)
          5k rows (Hierarchical Clustering)

Pipeline steps applied:
  1. Sentinel recoding   : 7/9/77/99/777/999 → NaN
  2. BMI conversion      : raw×100 → real BMI
  3. Feature dropping    : removed features with >40% missing
  4. Binary encoding     : BRFSS 1=yes/2=no → 1/0
  5. Diabetes collapsing : 4 categories → binary yes/no
  6. Winsorization       : continuous vars clipped at [1st, 99th] percentile
  7. Imputation          : median (continuous), mode (categorical)
  8. StandardScaler      : mean=0, std=1 per feature
  9. PCA                 : {n_components_90} components retaining ≥90% variance

Output matrices:
  X_scaled ({X30_scaled.shape[1]} features) — full feature matrix
  X_pca    ({X30_pca.shape[1]} components) — PCA-reduced matrix
  X_2d     (2 components)  — for visualization only

Feature list: {CLUSTER_COLS}
""")
print('=' * 65)

---
## ✅ End of Notebook 02 — Preprocessing

**Next step:** `03_kmeans.ipynb` — elbow plot, Silhouette curve, K-Means clustering.

Files produced:
- `data/X30_scaled.npy`, `data/X30_pca.npy`, `data/X30_2d.npy`
- `data/X5_scaled.npy`,  `data/X5_pca.npy`,  `data/X5_2d.npy`
- `data/scaler.pkl`, `data/pca.pkl`, `data/pca_2d.pkl`
- `data/demo_30k.csv`, `data/demo_5k.csv`
- `data/cluster_cols.json`
- `figures/scaling_comparison.png`
- `figures/pca_explained_variance.png`
- `figures/pca_loadings.png`